# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Comp5329_Assignment1_2026"

Mounted at /content/drive


In [ ]:

# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
PROJECT_ROOT = "E:\Working Area\Comp5329_Assignment1_2026"
#本地运行的位置

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\Admin\AppData\Local\Temp\ipykernel_31412\3064942964.py:1: SyntaxWarning: invalid escape sequence '\W'
  PROJECT_ROOT = "E:\Working Area\Comp5329_Assignment1_2026"


In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/Comp5329_Assignment1_2026


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:

from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 7000,
    checkpoint = 350,
    batch_size = 16,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:46<00:00,  4.34it/s]


STEP      200  loss 3302.148292



100%|██████████| 150/150 [00:09<00:00, 15.11it/s]


VALID(train) loss 70.192976  F1 6.972952  EM 0.000000



100%|██████████| 150/150 [00:09<00:00, 15.18it/s]


DEV         loss 68.681867  F1 7.614129  EM 0.041667

Learning rate: [0.001]
Saved best checkpoint to _model/model.pt (dev F1=7.6141, dev EM=0.0417)


 64%|██████▎   | 127/200 [00:28<00:16,  4.42it/s]


KeyboardInterrupt: 

---
## Section 4 - Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

---
## Section 5 - Experiments

This section groups the hypothesis experiments for H1 and H2.

### Section 5.0 - Shared Setup

Shared parameters for H1, H2, and later H3. Update this part first if you want the later experiment sections to use the same seed, paths, or training budget.

In [ ]:
# paths
train_npz_file = "_data/train.npz"
dev_npz_file = "_data/dev.npz"
word_emb_file = "_data/word_emb.json"
char_emb_file = "_data/char_emb.json"
train_eval_file = "_data/train_eval.json"
dev_eval_file = "_data/dev_eval.json"
word2idx_file = "_data/word2idx.json"
char2idx_file = "_data/char2idx.json"
raw_train_file = "_data/squad/train-mini.json"
raw_dev_file = "_data/squad/dev-v1.1.json"

# output folders
model_dir = "_model"
log_dir = "_log"

# experiment settings
seed = 42
batch_size = 16
num_steps = 8000
checkpoint = 500
early_stop = 3
lr = 1e-3
loss_name = "qa_nll"
activation = "relu"

# H1 will write the selected best combination here for H2 / H3
top_combos = []


### Section 5.1 - H1 Clean-Data Optimizer/Scheduler Comparison

Runs H1 in two stages on the clean train/dev split.

- Stage 1 uses a smaller local grid-search budget to compare all registered optimizer-scheduler combinations.
- Stage 2 reruns only the best combination with the full training budget from Section 5.0.
- `rank_by` switches the selection criterion between best dev F1 and best dev EM.
- The final full-trained best combination is saved to `top_combos` as a single-item list for H2 / H3.


#### Section 5.1.1 - H1 Local Search

Runs the reduced-budget search across all optimizer-scheduler combinations and stores the selected `best_combo` for the next block.


In [ ]:
from Optimizers import optimizers
from Schedulers import schedulers
from TrainTools.train import train

rank_by = "f1"  # "f1" or "em"

# Fixed local-search budget before the final full training run.
search_num_steps = 1000
search_checkpoint = 1000
search_val_num_batches = 100
search_dev_num_batches = 100

opt_list = list(optimizers.keys())
sched_list = list(schedulers.keys())
combos = [(opt_name, sched_name) for opt_name in opt_list for sched_name in sched_list]

h1_model_dir = f"{model_dir}/h1"
h1_log_dir = f"{log_dir}/h1"
h1_search_model_dir = f"{h1_model_dir}/search"
h1_search_log_dir = f"{h1_log_dir}/search"

def sort_rows(rows, rank_by):
    rows_f1 = sorted(rows, key=lambda row: (-row["best_dev_f1"], -row["best_dev_em"], row["combo_name"]))
    rows_em = sorted(rows, key=lambda row: (-row["best_dev_em"], -row["best_dev_f1"], row["combo_name"]))
    return rows_f1 if rank_by == "f1" else rows_em

search_train_args = {
    "train_npz": train_npz_file,
    "dev_npz": dev_npz_file,
    "word_emb_json": word_emb_file,
    "char_emb_json": char_emb_file,
    "train_eval_json": train_eval_file,
    "dev_eval_json": dev_eval_file,
    "batch_size": batch_size,
    "num_steps": search_num_steps,
    "checkpoint": search_checkpoint,
    "val_num_batches": search_val_num_batches,
    "dev_num_batches": search_dev_num_batches,
    "seed": seed,
    "learning_rate": lr,
    "loss_name": loss_name,
    "activation": activation,
}

rows = []

print("H1 local search budget:")
print({
    "search_num_steps": search_num_steps,
    "search_checkpoint": search_checkpoint,
    "search_val_num_batches": search_val_num_batches,
    "search_dev_num_batches": search_dev_num_batches,
})

for opt_name, sched_name in combos:
    combo_name = f"{opt_name}+{sched_name}"
    save_dir = f"{h1_search_model_dir}/{combo_name}"
    combo_log_dir = f"{h1_search_log_dir}/{combo_name}"

    print()
    print(f"=== Local search: {combo_name} ===")

    run_args = {
        **search_train_args,
        "optimizer_name": opt_name,
        "scheduler_name": sched_name,
        "save_dir": save_dir,
        "log_dir": combo_log_dir,
    }

    result = train(**run_args)
    best_row = max(result["history"], key=lambda row: (row["dev_f1"], row["dev_em"]))

    rows.append({
        "combo_name": combo_name,
        "optimizer_name": opt_name,
        "scheduler_name": sched_name,
        "best_dev_em": best_row["dev_em"],
        "best_dev_f1": best_row["dev_f1"],
        "search_step": best_row["step"],
    })

ranked_rows = sort_rows(rows, rank_by)
rank_label = "best dev F1" if rank_by == "f1" else "best dev EM"

print()
print(f"Ranking criterion: {rank_label}")
print(f"All combinations: {len(combos)}")

header = (
    f"{'rank':<6}"
    f"{'combo name':<24}"
    f"{'best step':>12}"
    f"{'best dev EM':>12}"
    f"{'best dev F1':>12}"
)
print()
print(header)
print("-" * len(header))

best_combo = ranked_rows[0]
h1_search_ranked_rows = ranked_rows

print(
    f"{1:<6}"
    f"{best_combo['combo_name']:<24}"
    f"{best_combo['search_step']:>12}"
    f"{best_combo['best_dev_em']:>12.4f}"
    f"{best_combo['best_dev_f1']:>12.4f}"
)
print()
print("H1 selected best combination:")
print(best_combo)


H1 local search budget:
{'search_num_steps': 1000, 'search_checkpoint': 1000, 'search_val_num_batches': 100, 'search_dev_num_batches': 100}

=== Local search: adam+cosine ===


100%|██████████| 1000/1000 [04:02<00:00,  4.12it/s]


STEP     1000  loss 43.040910



100%|██████████| 100/100 [00:06<00:00, 16.39it/s]


VALID(train) loss 4.660338  F1 7.534427  EM 1.312500



...

STEP     1000  loss 126.127147



100%|██████████| 100/100 [00:06<00:00, 16.25it/s]


VALID(train) loss 6.135416  F1 7.366303  EM 0.062500



100%|██████████| 100/100 [00:06<00:00, 16.40it/s]


DEV         loss 6.157376  F1 7.214299  EM 0.000000

Learning rate: [0.001]
Saved best checkpoint to _model/h1/search/sgd_momentum+lambda/model.pt (dev F1=7.2143, dev EM=0.0000)
Training finished.  Best F1: 7.2143  Best EM: 0.0000

Ranking criterion: best dev F1
All combinations: 9

rank  combo name                 best step best dev EM best dev F1
------------------------------------------------------------------
1     adam+lambda                     1000      4.8125      9.7577

H1 selected best combination:
{'combo_name': 'adam+lambda', 'optimizer_name': 'adam', 'scheduler_name': 'lambda', 'best_dev_em': 4.8125, 'best_dev_f1': 9.757668506015202, 'search_step': 1000}


#### Section 5.1.2 - H1 Best-Combo Full Training

Uses `best_combo` from Section 5.1.1 and reruns that single combination with the full training budget from Section 5.0. The resulting single-item `top_combos` list is reused by H2 / H3.


In [ ]:
from TrainTools.train import train

if "best_combo" not in globals():
    raise RuntimeError("Run Section 5.1.1 first so `best_combo` is available.")

h1_model_dir = f"{model_dir}/h1"
h1_log_dir = f"{log_dir}/h1"

full_train_args = {
    "train_npz": train_npz_file,
    "dev_npz": dev_npz_file,
    "word_emb_json": word_emb_file,
    "char_emb_json": char_emb_file,
    "train_eval_json": train_eval_file,
    "dev_eval_json": dev_eval_file,
    "batch_size": batch_size,
    "num_steps": num_steps,
    "checkpoint": checkpoint,
    "seed": seed,
    "early_stop": early_stop,
    "learning_rate": lr,
    "loss_name": loss_name,
    "activation": activation,
}

print(f"=== Full training: {best_combo['combo_name']} ===")

best_model_result = train(
    **full_train_args,
    optimizer_name=best_combo["optimizer_name"],
    scheduler_name=best_combo["scheduler_name"],
    save_dir=f"{h1_model_dir}/{best_combo['combo_name']}",
    log_dir=f"{h1_log_dir}/{best_combo['combo_name']}",
)
best_model_row = max(best_model_result["history"], key=lambda row: (row["dev_f1"], row["dev_em"]))

top_combos = [best_combo["combo_name"]]
h1_best_result = {
    "combo_name": best_combo["combo_name"],
    "optimizer_name": best_combo["optimizer_name"],
    "scheduler_name": best_combo["scheduler_name"],
    "best_dev_em": best_model_row["dev_em"],
    "best_dev_f1": best_model_row["dev_f1"],
    "ckpt_path": best_model_result["ckpt_path"],
}

print()
print("H1 selected combination for H2 / H3:")
print(top_combos)
print()
print("H1 full-training result:")
print(h1_best_result)


=== Full training: adam+lambda ===


100%|██████████| 500/500 [02:03<00:00,  4.05it/s]


STEP      500  loss 81.106802



100%|██████████| 150/150 [00:09<00:00, 16.17it/s]


VALID(train) loss 4.677817  F1 8.016151  EM 0.000000



...

STEP     3500  loss 4.026160



100%|██████████| 150/150 [00:09<00:00, 16.28it/s]


VALID(train) loss 3.652778  F1 9.939691  EM 1.958333



100%|██████████| 150/150 [00:09<00:00, 16.20it/s]


DEV         loss 4.536909  F1 9.393047  EM 0.875000

Learning rate: [0.001]
Early stopping triggered.
Training finished.  Best F1: 11.4978  Best EM: 4.9167

H1 selected combination for H2 / H3:
['adam+lambda']

H1 full-training result:
{'combo_name': 'adam+lambda', 'optimizer_name': 'adam', 'scheduler_name': 'lambda', 'best_dev_em': 4.916666666666667, 'best_dev_f1': 11.497782221839671, 'ckpt_path': '/content/drive/MyDrive/Comp5329_Assignment1_2026/_model/h1/adam+lambda/model.pt'}


### Section 5.2 - H2 Corrupted Develop Data

Builds two label-preserving noisy dev sets from the original clean dev set.

- `question typo noise`: applies one mild character swap to one non-trivial question word.
- `context distractor noise`: appends one irrelevant sentence to the end of the context.
- The answer labels stay usable because the answer span in the original context is untouched.


#### Section 5.2.1 - Data Process

In [ ]:
import os
import re
import random
from collections import Counter
from copy import deepcopy

import ujson as json

from Tools.preproc import process_file, build_features

h2_data_dir = "_data/h2"
typo_json = f"{h2_data_dir}/dev_typo.json"
typo_npz = f"{h2_data_dir}/dev_typo.npz"
typo_eval = f"{h2_data_dir}/dev_typo_eval.json"

distractor_json = f"{h2_data_dir}/dev_distractor.json"
distractor_npz = f"{h2_data_dir}/dev_distractor.npz"
distractor_eval = f"{h2_data_dir}/dev_distractor_eval.json"

os.makedirs(h2_data_dir, exist_ok=True)

with open(raw_dev_file, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(word2idx_file, "r", encoding="utf-8") as f:
    word2idx = json.load(f)

with open(char2idx_file, "r", encoding="utf-8") as f:
    char2idx = json.load(f)

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f)

def typo_word(word):
    chars = list(word)
    chars[1], chars[2] = chars[2], chars[1]
    return "".join(chars)

def add_typo(question, rng):
    matches = list(re.finditer(r"[A-Za-z]{4,}", question))
    if not matches:
        return question
    target = matches[rng.randrange(len(matches))]
    new_word = typo_word(target.group(0))
    return question[:target.start()] + new_word + question[target.end():]

distractor_sent = "An unrelated report later discussed a different city and year, but this sentence does not answer the question."

def add_distractor(context):
    context = context.rstrip()
    if context and context[-1] not in ".!?":
        context += "."
    return context + " " + distractor_sent

def build_noisy_files(json_file, npz_file, eval_file):
    examples, eval_data = process_file(json_file, "dev", Counter(), Counter())
    build_features(examples, "dev", npz_file, word2idx, char2idx, 400, 50, 30, 16)
    save_json(eval_file, eval_data)

rng = random.Random(seed)

typo_data = deepcopy(dev_data)
for article in typo_data["data"]:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            qa["question"] = add_typo(qa["question"], rng)

distractor_data = deepcopy(dev_data)
for article in distractor_data["data"]:
    for para in article["paragraphs"]:
        para["context"] = add_distractor(para["context"])

save_json(typo_json, typo_data)
save_json(distractor_json, distractor_data)
build_noisy_files(typo_json, typo_npz, typo_eval)
build_noisy_files(distractor_json, distractor_npz, distractor_eval)

print("H2 noisy dev files are ready under:", h2_data_dir)


#### Section 5.2.2 - H2 Selected Combinations

By default this reuses `top_combos` from Section 5.1. If you want to run H2 independently, overwrite `selected_combos` in this cell.

In [ ]:
selected_combos = list(top_combos)

# If you want to run H2 independently, overwrite it here.
# selected_combos = [
#     "sgd+step",
#     "sgd+cosine",
#     "adam+lambda",
# ]

print("H2 selected combinations:")
print(selected_combos)


H2 selected combinations:
['adam+lambda']


#### Section 5.2.3 - H2 Noisy Evaluation

Uses the clean-trained checkpoints from H1 and evaluates each selected combination on clean dev, typo-noisy dev, and distractor-noisy dev.

In [ ]:
from EvaluateTools.evaluate import evaluate

h2_model_dir = f"{model_dir}/h1"
h2_log_dir = f"{log_dir}/h2"

def pair(metrics):
    return f"{metrics['exact_match']:.2f}/{metrics['f1']:.2f}"

def drop(clean_metrics, noisy_metrics):
    em_drop = clean_metrics["exact_match"] - noisy_metrics["exact_match"]
    f1_drop = clean_metrics["f1"] - noisy_metrics["f1"]
    return f"{em_drop:.2f}/{f1_drop:.2f}"

rows = []

for combo_name in selected_combos:
    save_dir = f"{h2_model_dir}/{combo_name}"

    clean_metrics = evaluate(
        dev_npz=dev_npz_file,
        word_emb_json=word_emb_file,
        char_emb_json=char_emb_file,
        dev_eval_json=dev_eval_file,
        save_dir=save_dir,
        log_dir=f"{h2_log_dir}/{combo_name}/clean",
        batch_size=batch_size,
        loss_name=loss_name,
    )

    typo_metrics = evaluate(
        dev_npz=typo_npz,
        word_emb_json=word_emb_file,
        char_emb_json=char_emb_file,
        dev_eval_json=typo_eval,
        save_dir=save_dir,
        log_dir=f"{h2_log_dir}/{combo_name}/typo",
        batch_size=batch_size,
        loss_name=loss_name,
    )

    distractor_metrics = evaluate(
        dev_npz=distractor_npz,
        word_emb_json=word_emb_file,
        char_emb_json=char_emb_file,
        dev_eval_json=distractor_eval,
        save_dir=save_dir,
        log_dir=f"{h2_log_dir}/{combo_name}/distractor",
        batch_size=batch_size,
        loss_name=loss_name,
    )

    rows.append({
        "combo_name": combo_name,
        "clean_metrics": clean_metrics,
        "typo_metrics": typo_metrics,
        "distractor_metrics": distractor_metrics,
    })

header = (
    f"{'combo name':<24}"
    f"{'clean EM/F1':>16}"
    f"{'typo EM/F1':>16}"
    f"{'distractor EM/F1':>22}"
    f"{'typo drop':>14}"
    f"{'distractor drop':>20}"
)
print()
print(header)
print("-" * len(header))

for row in rows:
    clean_pair = pair(row["clean_metrics"])
    typo_pair = pair(row["typo_metrics"])
    distractor_pair = pair(row["distractor_metrics"])
    typo_drop = drop(row["clean_metrics"], row["typo_metrics"])
    distractor_drop = drop(row["clean_metrics"], row["distractor_metrics"])
    print(
        f"{row['combo_name']:<24}"
        f"{clean_pair:>16}"
        f"{typo_pair:>16}"
        f"{distractor_pair:>22}"
        f"{typo_drop:>14}"
        f"{distractor_drop:>20}"
    )


### Section 5.3 - H3 Noise-Augmented Training

Compares clean-only training and typo-augmented training for the selected top-k combinations.

- H3 keeps the optimizer-scheduler combinations fixed.
- H3 uses question typo augmentation on a subset of training samples.
- H3 evaluates robustness on the typo-noisy dev set.


#### Section 5.3.1 - Data Process


In [ ]:
import os
import re
import random
from collections import Counter
from copy import deepcopy

import ujson as json

from Tools.preproc import process_file, build_features

h3_data_dir = "_data/h3"
train_typo_json = f"{h3_data_dir}/train_typo.json"
train_typo_npz = f"{h3_data_dir}/train_typo.npz"
dev_typo_json = f"{h3_data_dir}/dev_typo.json"
dev_typo_npz = f"{h3_data_dir}/dev_typo.npz"
dev_typo_eval = f"{h3_data_dir}/dev_typo_eval.json"
aug_ratio = 0.3

os.makedirs(h3_data_dir, exist_ok=True)

with open(raw_train_file, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(raw_dev_file, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(word2idx_file, "r", encoding="utf-8") as f:
    word2idx = json.load(f)

with open(char2idx_file, "r", encoding="utf-8") as f:
    char2idx = json.load(f)

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f)

def typo_word(word):
    chars = list(word)
    chars[1], chars[2] = chars[2], chars[1]
    return "".join(chars)

def add_typo(question, rng):
    matches = list(re.finditer(r"[A-Za-z]{4,}", question))
    if not matches:
        return question
    target = matches[rng.randrange(len(matches))]
    new_word = typo_word(target.group(0))
    return question[:target.start()] + new_word + question[target.end():]

def build_file(json_file, data_type, npz_file, eval_file=None):
    examples, eval_data = process_file(json_file, data_type, Counter(), Counter())
    build_features(examples, data_type, npz_file, word2idx, char2idx, 400, 50, 30, 16)
    if eval_file is not None:
        save_json(eval_file, eval_data)

rng = random.Random(seed)

train_typo_data = deepcopy(train_data)
for article in train_typo_data["data"]:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            if rng.random() < aug_ratio:
                qa["question"] = add_typo(qa["question"], rng)

dev_typo_data = deepcopy(dev_data)
for article in dev_typo_data["data"]:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            qa["question"] = add_typo(qa["question"], rng)

save_json(train_typo_json, train_typo_data)
save_json(dev_typo_json, dev_typo_data)
build_file(train_typo_json, "train", train_typo_npz)
build_file(dev_typo_json, "dev", dev_typo_npz, dev_typo_eval)

print("H3 typo train/dev files are ready under:", h3_data_dir)


Generating train examples…


100%|██████████| 150/150 [00:04<00:00, 35.83it/s]


  30293 questions in total
Processing train examples…


100%|██████████| 30293/30293 [00:07<00:00, 4017.94it/s]


  Built 30169 / 30293 instances
Generating dev examples…


100%|██████████| 48/48 [00:01<00:00, 37.55it/s]


  10570 questions in total
Processing dev examples…


100%|██████████| 10570/10570 [00:02<00:00, 3628.18it/s]


  Built 10465 / 10570 instances
H3 typo train/dev files are ready under: _data/h3


#### Section 5.3.2 - H3 Selected Combinations

By default this reuses `top_combos` from Section 5.1. If you want to run H3 independently, overwrite `h3_combos` in this cell.

In [ ]:
h3_combos = list(top_combos)

# If you want to run H3 independently, overwrite it here.
# h3_combos = [
#     "sgd+step",
#     "sgd+cosine",
#     "adam+lambda",
# ]

print("H3 selected combinations:")
print(h3_combos)


#### Section 5.3.3 - H3 Train and Evaluation

Trains each selected combination with clean-only training and typo-augmented training, then compares clean and noisy evaluation results.

In [ ]:
from TrainTools.train import train
from EvaluateTools.evaluate import evaluate

h3_model_dir = f"{model_dir}/h3"
h3_log_dir = f"{log_dir}/h3"
train_modes = ["clean", "aug"]

def pair(metrics):
    return f"{metrics['exact_match']:.2f}/{metrics['f1']:.2f}"

def drop(clean_metrics, noisy_metrics):
    em_drop = clean_metrics["exact_match"] - noisy_metrics["exact_match"]
    f1_drop = clean_metrics["f1"] - noisy_metrics["f1"]
    return f"{em_drop:.2f}/{f1_drop:.2f}"

h3_train_args = {
    "dev_npz": dev_npz_file,
    "word_emb_json": word_emb_file,
    "char_emb_json": char_emb_file,
    "train_eval_json": train_eval_file,
    "dev_eval_json": dev_eval_file,
    "batch_size": batch_size,
    "num_steps": num_steps,
    "checkpoint": checkpoint,
    "seed": seed,
    "early_stop": early_stop,
    "learning_rate": lr,
    "loss_name": loss_name,
    "activation": activation,
}

rows = []

for combo_name in h3_combos:
    opt_name, sched_name = combo_name.split("+")

    for train_mode in train_modes:
        train_npz = train_npz_file if train_mode == "clean" else train_typo_npz
        save_dir = f"{h3_model_dir}/{train_mode}/{combo_name}"
        train_log_dir = f"{h3_log_dir}/{train_mode}/{combo_name}/train"

        run_args = {
            **h3_train_args,
            "train_npz": train_npz,
            "optimizer_name": opt_name,
            "scheduler_name": sched_name,
            "save_dir": save_dir,
            "log_dir": train_log_dir,
        }

        train(**run_args)

        clean_metrics = evaluate(
            dev_npz=dev_npz_file,
            word_emb_json=word_emb_file,
            char_emb_json=char_emb_file,
            dev_eval_json=dev_eval_file,
            save_dir=save_dir,
            log_dir=f"{h3_log_dir}/{train_mode}/{combo_name}/clean",
            batch_size=batch_size,
            loss_name=loss_name,
        )

        noisy_metrics = evaluate(
            dev_npz=dev_typo_npz,
            word_emb_json=word_emb_file,
            char_emb_json=char_emb_file,
            dev_eval_json=dev_typo_eval,
            save_dir=save_dir,
            log_dir=f"{h3_log_dir}/{train_mode}/{combo_name}/noisy",
            batch_size=batch_size,
            loss_name=loss_name,
        )

        rows.append({
            "combo_name": combo_name,
            "train_mode": train_mode,
            "clean_metrics": clean_metrics,
            "noisy_metrics": noisy_metrics,
        })

header = (
    f"{'combo name':<24}"
    f"{'training mode':<14}"
    f"{'clean EM/F1':>16}"
    f"{'noisy EM/F1':>16}"
    f"{'robustness drop':>18}"
)
print()
print(header)
print("-" * len(header))

for row in rows:
    clean_pair = pair(row["clean_metrics"])
    noisy_pair = pair(row["noisy_metrics"])
    noisy_drop = drop(row["clean_metrics"], row["noisy_metrics"])
    print(
        f"{row['combo_name']:<24}"
        f"{row['train_mode']:<14}"
        f"{clean_pair:>16}"
        f"{noisy_pair:>16}"
        f"{noisy_drop:>18}"
    )


100%|██████████| 500/500 [02:02<00:00,  4.08it/s]


STEP      500  loss 81.106802



100%|██████████| 150/150 [00:09<00:00, 16.28it/s]


VALID(train) loss 4.677817  F1 8.016151  EM 0.000000



100%|██████████| 150/150 [00:09<00:00, 16.40it/s]


...

DEV         loss 5.145123  F1 22.331635  EM 14.541667

Learning rate: [0.001]
Saved best checkpoint to _model/h3/aug/adam+lambda/model.pt (dev F1=22.3316, dev EM=14.5417)
Training finished.  Best F1: 22.3316  Best EM: 14.5417


100%|██████████| 655/655 [00:40<00:00, 16.37it/s]


DEV   loss 5.049525  F1 20.267268  EM 11.763020


100%|██████████| 655/655 [00:39<00:00, 16.52it/s]


DEV   loss 5.195513  F1 19.444283  EM 11.141902

combo name              training mode      clean EM/F1     noisy EM/F1   robustness drop
----------------------------------------------------------------------------------------
adam+lambda             clean               3.65/10.34      3.65/10.37        0.00/-0.03
adam+lambda             aug                11.76/20.27     11.14/19.44         0.62/0.82
